### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [122]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [123]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [124]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [125]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [126]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 25% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.25, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [127]:
####################### A function to calculate the collateral ########################
def calculate_collateral(profession):
    if "HighSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        # random.random() returns a float number between 0 and 1
        if random.random() < 0.10: # 10 % of the people do not have collateral
            return 0
        else: # 90 % have a collateral between 10,000 and 80,000
            return random.randint(10000, 80000)
    elif "MediumSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        if random.random() < 0.20: # 20 % of the people do not have collateral
            return 0
        else: # 80 % have a collateral between 10,000 and 50,000
            return random.randint(10000, 50000)
    else: # If Lowskilled
        if random.random() < 0.35: # 35 % of the people do not have collateral
            return 0
        else: # 65 % have a collateral between 5,000 and 25,000
            return random.randint(5000, 25000)

In [128]:
####################### A function to estimate the seizable assets to calculate the LGD ########################
def estimate_seizable_assets(monthly_income, savings, profession, collateral):
    
    # Base asset estimation as a portion of income and savings
    # This is a proxy, people with higher income, tend to have higher assets
    # If the savinds are negative, and higher than 2 times the monthy income this will draw this to be negative
    # In the end, if no assets can be seizured, it means that maybe it is not convenient for the bank to actually give the loan
    base_asset = savings + 2 * monthly_income

    # People that are High-Skilled tend to have more assets to be seized, even if they are currently unemployed
    if "HighSkilled" in profession:
        base_asset *= 1.2
    elif "MediumSkilled" in profession:
        base_asset *= 1.0
    else:
        base_asset *= 0.8

    # If they have a collateral, more sizes are
    base_asset += collateral

    # Add some noise to simulate unpredictability
    # That means given some processes in normal life, it is not always sure that 100% of the collateral can be recovered without cost
    base_asset *= np.random.normal(1, 0.1)  # 10% variation

    # To ensure that if nothing can be seized, because in the end a negative value is returned, then we get a zero
    return max(base_asset, 0)

In [129]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

##### Data Generator for the original state of individuals

In [130]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the requested duration of the loan (maybe we can make it to be then also set by the bank whether it accepts it up to this term or not)
        if 0.25 <= credit_to_income_ratio < 0.5: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.2, 0.3, 0.2, 0.2, 0.01], k=1)[0]
        elif 0.5 <= credit_to_income_ratio < 0.75: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.1, 0.2, 0.2, 0.2, 0.3], k=1)[0]
        elif 0.75 <= credit_to_income_ratio < 1.0: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.08, 0.12, 0.2, 0.25, 0.35], k=1)[0]
        else:
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.02, 0.08, 0.2, 0.2, 0.5], k=1)[0]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
            
        # collateral and seizurable asset
        collateral = calculate_collateral(profession)
        estimated_seizable_assets = estimate_seizable_assets(monthly_income, savings_debt, profession, collateral)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'requested_loan_duration': credit_term_months, # depends on the credit-to-income ratio
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'collateral': collateral, # depends on the profession
            'estimated seizable assets': estimated_seizable_assets, # it is based on the monthly income, the savings, the profession and the collateral
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [131]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
0,name0,55,bachelor degree,1,1,HighSkilled,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,48,0.273274,10286,22213.991840,0
1,name1,43,ausbildung,0,3,MediumSkilled,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,24,1.993925,33596,35569.694991,1
2,name2,46,ausbildung,0,0,MediumSkilled,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,36,0.122974,47808,44913.626587,0
3,name3,54,bachelor degree,0,0,HighSkilled,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,48,0.273364,65197,73048.375602,0
4,name4,36,post graduate degree,1,0,HighSkilled,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,24,0.957580,0,6506.227319,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,high school or lower,1,1,LowSkilled,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,36,0.654171,17796,21359.515656,0
996,name996,31,high school or lower,1,0,LowSkilled,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,36,3.163049,0,91.300003,1
997,name997,59,ausbildung,3,1,MediumSkilled,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,36,0.855348,30301,40849.559109,1
998,name998,32,ausbildung,2,0,MediumSkilled,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,60,0.847765,31155,34890.802857,0


##### DF statistics

In [132]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.160000,0.490000,0.563000,3483.544000,3208.436846,275.107154,0.128476,3045.408715,0.863071,41.652000,0.845029,25696.412000,33469.095155,0.503000
std,8.892327,0.698848,0.823838,2572.014586,2442.556265,1666.772375,0.264127,2796.832486,0.361949,16.290865,0.499962,21419.236325,25418.158449,0.500241
min,30.000000,0.000000,0.000000,500.000000,397.434871,-7282.707532,0.000000,191.786562,0.250100,12.000000,0.000000,0.000000,91.300003,0.000000
25%,38.000000,0.000000,0.000000,1665.750000,1490.876937,-312.978644,0.000000,1259.758203,0.533992,24.000000,0.467627,8659.000000,13020.198067,0.000000
50%,45.000000,0.000000,0.000000,2583.000000,2410.963129,221.829460,0.000000,2121.094317,0.859688,48.000000,0.835855,21545.000000,27816.199018,1.000000
75%,53.000000,1.000000,1.000000,4500.000000,4145.065571,739.280135,0.158281,3695.929214,1.173887,60.000000,1.154614,40281.500000,49273.597159,1.000000
max,60.000000,3.000000,4.000000,14316.000000,17497.177007,11257.112312,2.220228,20405.717420,1.499850,60.000000,3.163049,79605.000000,119287.601837,1.000000


Monthly income by profession

In [133]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,6117.454006,2674.063117,1650.0,4196.0,5607.0,7512.0,14316.0
LowSkilled,283.0,1578.176678,533.828435,500.0,1166.0,1497.0,2011.5,3031.0
MediumSkilled,289.0,2785.079585,968.415819,950.0,2069.0,2625.0,3402.0,6401.0
Unemployed_HighSkilled,28.0,3223.214286,970.250268,1500.0,2250.0,3250.0,4000.0,4500.0
Unemployed_LowSkilled,28.0,857.142857,208.927724,500.0,712.5,900.0,1000.0,1100.0
Unemployed_MediumSkilled,35.0,1605.714286,351.210392,900.0,1300.0,1750.0,2000.0,2000.0


Debt-to-income ratio by profession

In [134]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,0.135398,0.280133,0.0,0.0,0.0,0.165025,1.566623
LowSkilled,283.0,0.129426,0.262116,0.0,0.0,0.0,0.148907,1.769135
MediumSkilled,289.0,0.140923,0.279697,0.0,0.0,0.0,0.173111,2.220228
Unemployed_HighSkilled,28.0,0.077872,0.104346,0.0,0.0,0.0,0.142963,0.352629
Unemployed_LowSkilled,28.0,0.070843,0.120073,0.0,0.0,0.0,0.107376,0.401235
Unemployed_MediumSkilled,35.0,0.037963,0.081287,0.0,0.0,0.0,0.000000,0.276710


In [135]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,0.848196,0.512556,0.000000,0.442305,0.870733,1.160763,2.812955
LowSkilled,283.0,0.821859,0.523461,0.000000,0.422807,0.772479,1.146979,3.163049
MediumSkilled,289.0,0.873902,0.493353,0.000000,0.548242,0.858648,1.180756,2.851557
Unemployed_HighSkilled,28.0,0.817526,0.492350,0.000000,0.427651,0.738453,1.250184,1.744338
Unemployed_LowSkilled,28.0,0.833699,0.366473,0.139014,0.510811,0.851827,1.171378,1.447993
Unemployed_MediumSkilled,35.0,0.794527,0.315719,0.263496,0.594907,0.795685,1.048908,1.419000


In [136]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,0.860063,0.366819,0.250199,0.516490,0.860126,1.177106,1.499850
LowSkilled,283.0,0.839670,0.369718,0.253037,0.499953,0.831492,1.158614,1.498852
MediumSkilled,289.0,0.880508,0.348580,0.250100,0.579912,0.909411,1.181173,1.499232
Unemployed_HighSkilled,28.0,0.914771,0.382497,0.259387,0.548627,1.041764,1.212261,1.497020
Unemployed_LowSkilled,28.0,0.757393,0.322029,0.351106,0.460064,0.724942,1.012086,1.391502
Unemployed_MediumSkilled,35.0,0.980464,0.353880,0.253388,0.715093,1.035293,1.231504,1.493661


In [137]:
df_1.groupby('profession')['collateral'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,40343.646884,22995.736025,0.0,22489.00,41582.0,60230.00,78987.0
LowSkilled,283.0,9427.491166,8356.982608,0.0,0.00,9364.0,16475.00,24979.0
MediumSkilled,289.0,24071.394464,15841.736729,0.0,12553.00,25227.0,37053.00,49960.0
Unemployed_HighSkilled,28.0,46934.428571,19650.487025,10925.0,34229.25,45886.5,63432.75,79605.0
Unemployed_LowSkilled,28.0,9805.750000,9405.193891,0.0,0.00,8344.5,17345.50,24448.0
Unemployed_MediumSkilled,35.0,25350.428571,15520.206899,0.0,16525.50,26471.0,38564.50,49058.0


In [138]:
df_1.groupby('profession')['estimated seizable assets'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,55090.163456,24780.225718,2071.131126,36916.116260,54638.147434,74796.131623,119287.601837
LowSkilled,283.0,12210.626907,8697.666976,91.300003,3121.016783,11979.790553,19482.718811,30960.999255
MediumSkilled,289.0,29862.669682,16619.548397,456.229581,17284.205067,30643.145440,42289.719651,69287.585477
Unemployed_HighSkilled,28.0,54201.711358,19752.978765,15071.084459,44148.154656,52178.261049,70569.454453,92109.434527
Unemployed_LowSkilled,28.0,10787.438422,8982.449400,876.729756,1691.936536,10319.776821,17705.616036,25711.826703
Unemployed_MediumSkilled,35.0,28516.997818,16295.738779,2452.800266,17099.308172,29577.362543,43290.738987,54170.466460


In [139]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,337.0,0.575668,0.494976,0.0,0.0,1.0,1.0,1.0
LowSkilled,283.0,0.508834,0.500808,0.0,0.0,1.0,1.0,1.0
MediumSkilled,289.0,0.404844,0.491713,0.0,0.0,0.0,1.0,1.0
Unemployed_HighSkilled,28.0,0.428571,0.503953,0.0,0.0,0.0,1.0,1.0
Unemployed_LowSkilled,28.0,0.642857,0.487950,0.0,0.0,1.0,1.0,1.0
Unemployed_MediumSkilled,35.0,0.514286,0.507093,0.0,0.0,1.0,1.0,1.0


# MODELLING PD, LGD, EAD

## PD (probability of default)
The probability that a customer with default at some point

#### Packages

In [140]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Converting variables into dummies

In [141]:
# Convert categorical variables to numeric
df_R = pd.get_dummies(df_1, columns=["educational level", "profession"], drop_first=True)
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,estimated seizable assets,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled
0,name0,55,1,1,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,...,22213.991840,0,True,False,False,False,False,False,False,False
1,name1,43,0,3,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,...,35569.694991,1,False,False,False,False,True,False,False,False
2,name2,46,0,0,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,...,44913.626587,0,False,False,False,False,True,False,False,False
3,name3,54,0,0,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,...,73048.375602,0,True,False,False,False,False,False,False,False
4,name4,36,1,0,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,...,6506.227319,1,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,1,1,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,...,21359.515656,0,False,True,False,True,False,False,False,False
996,name996,31,1,0,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,...,91.300003,1,False,True,False,True,False,False,False,False
997,name997,59,3,1,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,...,40849.559109,1,False,False,False,False,True,False,False,False
998,name998,32,2,0,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,...,34890.802857,0,False,False,False,False,True,False,False,False


#### Defining X and y

In [142]:
# Column "name" is dropped from the dataframe, no need to keep it
# All the variables except y-categorical-default are X
X = df_R.drop(columns=["name", "y-categorical-default"])
y = df_R["y-categorical-default"]

#### Split between trainning and test sets

In [143]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Standardize the variables for a better gradient descendt
Standardizing features to have a mean of zero ensures that all features are centered around the same baseline, which helps prevent models from being biased toward features with larger numerical values. It also makes gradient-based optimization methods like gradient descent behave more efficiently by ensuring all features contribute equally to the cost function.

In [144]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LOGIT

##### Packages

In [145]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Training the Logistic Regression

In [146]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

##### Predictions

In [147]:
y_pred = log_reg.predict(X_test_scaled)

##### Evaluations of the accuracy of model

In [148]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.64


In [149]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.65      0.64        99
           1       0.65      0.63      0.64       101

    accuracy                           0.64       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.64      0.64      0.64       200



##### Estimating the PDs

In [150]:
df_R["PD_LR"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
df_R


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR
0,name0,55,1,1,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,...,0,True,False,False,False,False,False,False,False,0.679787
1,name1,43,0,3,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,...,1,False,False,False,False,True,False,False,False,0.701779
2,name2,46,0,0,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,...,0,False,False,False,False,True,False,False,False,0.125340
3,name3,54,0,0,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,...,0,True,False,False,False,False,False,False,False,0.464511
4,name4,36,1,0,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,...,1,False,False,True,False,False,False,False,False,0.869159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,1,1,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,...,0,False,True,False,True,False,False,False,False,0.642825
996,name996,31,1,0,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,...,1,False,True,False,True,False,False,False,False,0.987255
997,name997,59,3,1,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,...,1,False,False,False,False,True,False,False,False,0.892420
998,name998,32,2,0,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,...,0,False,False,False,False,True,False,False,False,0.830169


### Using Neuronal Networks

##### Packages

In [151]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

##### Building the Neuronal Network

In [152]:
# We may try out:

# tanh: The hyperbolic tangent function outputs values between -1 and 1, making it useful for hidden layers 
# where you want activations that are zero-centered, which helps in faster convergence and avoids saturation for small inputs.
 
# relu: The Rectified Linear Unit activation function outputs zero for any negative input and passes positive values as they are.
# It is widely used in hidden layers for its simplicity and effectiveness, and helps avoid the vanishing gradient problem seen with functions like sigmoid and tanh.

# softmax: Softmax is typically used in the output layer for multi-class classification tasks. It converts the raw outputs into probabilities, 
# ensuring that the sum of all output values equals 1, representing the probability distribution over multiple classes.

# This is like having an input which is your variable X, then 32 neurons process the input features,
# using a function (in this case "tanh") to calculate the weights and transformations at each neuron. 
# The results are then passed to a subsequent layer with 16 neurons, where again "tanh" is applied to further transform the data.
# In the end, everything is passed through a final neuron that uses a "sigmoid" (logistic) function 
# to produce an output between [0, 1], representing a probability for binary classification.
model = Sequential([  # Each layer is run after the other, forming a linear stack of layers.
    # The first Dense layer applies 32 units (neurons) and uses the "tanh" activation function.
    # The input_shape corresponds to the number of features in the dataset (X_train_scaled).
    Dense(32, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    
    # The second Dense layer applies 16 units (neurons) and uses "tanh" activation function.
    # "tanh" ensures that the output of each neuron will be between -1 and 1, centering the activations.
    Dense(16, activation='tanh'),
    
    # The final Dense layer outputs a single value, which is the probability of the positive class.
    # Sigmoid activation squashes the output to a value between 0 and 1.
    # This is commonly used for binary classification, where the output is a probability of class 1.
    Dense(1, activation='sigmoid')
])

/Users/bonjour/opt/anaconda3/envs/bankgame/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##### Compiling the model

In [153]:
# The model is being compiled with the following parameters:
# optimizer='adam': The Adam optimizer is being used. It is an adaptive learning rate optimization algorithm that 
#  combines the benefits of both AdaGrad and RMSProp, making it well-suited for most deep learning models.
# loss='binary_crossentropy': The loss function used is binary cross-entropy, which is appropriate for binary classification 
#  tasks where the output is a probability of belonging to one of two classes, which is the case of our y-variable
# metrics=['accuracy']: The model will track accuracy as the evaluation metric during training and testing, 
#  which measures the percentage of correct predictions.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

##### Training the model

In [154]:
# The model is fit with X_train_scaled: This means the model is being trained on the scaled training data (X_train_scaled) 
# using the corresponding labels (y_train).

# Uses 25 epochs: An epoch refers to one full pass through the entire training dataset. 
# The model will train for 25 epochs, meaning it will go through the data 25 times to learn the optimal weights.

# It will use a batch size of 32: The model will train using 32 samples (or rows of data) at a time, and after processing 
# those 32, it updates the weights before moving on to the next 32 samples. 

# During training, the model's performance is periodically evaluated on the validation set (X_test_scaled and y_test) 
# to monitor overfitting and to adjust the training accordingly.

model_NN = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5783 - loss: 0.6757 - val_accuracy: 0.6250 - val_loss: 0.6575
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6193 - loss: 0.6336 - val_accuracy: 0.6650 - val_loss: 0.6351
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6485 - loss: 0.6098 - val_accuracy: 0.6700 - val_loss: 0.6243
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6758 - loss: 0.5962 - val_accuracy: 0.6650 - val_loss: 0.6160
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6709 - loss: 0.6013 - val_accuracy: 0.6650 - val_loss: 0.6189
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6827 - loss: 0.5881 - val_accuracy: 0.6550 - val_loss: 0.6172
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6572 - loss: 0.5863 - val_accuracy: 0.6500 - val_loss: 0.6182
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6813 - loss: 0.5900 - val_accuracy: 0.6700 - val_loss:

##### We want to get the last accuracy of the last epoch and the minimal and highest accuracy values for comparison

In [155]:
train_accuracies_NN = model_NN.history['accuracy']

In [156]:
# final accuracy value
final_accuracy = train_accuracies_NN[-1]
# maximal accuracy value
min_accuracy = min(train_accuracies_NN)
# minimal accuracy value
max_accuracy = max(train_accuracies_NN)
# average accuracy value
average_accuracy = sum(train_accuracies_NN) / len(train_accuracies_NN)

In [157]:
# Print the results
print(f'Final accuracy: {final_accuracy:.4f}')
print(f'Minimum accuracy during training of the NN: {min_accuracy:.4f}')
print(f'Maximum accuracy during training of the NN: {max_accuracy:.4f}')
print(f'Average accuracy during training of the NN: {average_accuracy:.4f}')

Final accuracy: 0.7100
Minimum accuracy during training of the NN: 0.5825
Maximum accuracy during training of the NN: 0.7100
Average accuracy during training of the NN: 0.6811


##### Estimating the PDs

In [158]:
df_R["PD_NN"]= model.predict(scaler.transform(X))
df_R

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN
0,name0,55,1,1,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,...,True,False,False,False,False,False,False,False,0.679787,0.712740
1,name1,43,0,3,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,...,False,False,False,False,True,False,False,False,0.701779,0.707140
2,name2,46,0,0,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,...,False,False,False,False,True,False,False,False,0.125340,0.065778
3,name3,54,0,0,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,...,True,False,False,False,False,False,False,False,0.464511,0.419346
4,name4,36,1,0,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,...,False,False,True,False,False,False,False,False,0.869159,0.932222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,1,1,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,...,False,True,False,True,False,False,False,False,0.642825,0.705934
996,name996,31,1,0,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,...,False,True,False,True,False,False,False,False,0.987255,0.946485
997,name997,59,3,1,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,...,False,False,False,False,True,False,False,False,0.892420,0.775484
998,name998,32,2,0,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,...,False,False,False,False,True,False,False,False,0.830169,0.857296


### Using Random Forests

##### Packages

In [159]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##### Fitting the model

In [160]:
# Initialize the RandomForestClassifier with the following parameters:
# n_estimators=100: This sets the number of decision trees (estimators) in the forest. The model will train 100 individual trees and aggregate their results to make predictions.
# random_state=42: This ensures reproducibility by fixing the random seed used in the training process. 
# To get the same result even if the code is run multiple times (obviously this will only be affected by the random nature of our data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

##### Accuracy of the model

In [161]:
# Predict class labels for X_test_scaled
y_pred = rf_model.predict(X_test_scaled)
# Calculate accuracy by comparing the predicted labels with our simulated data y
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.6850


##### Estimating the PDs

In [162]:
# df_R = df_R.iloc[:len(X_test_scaled)]
# df_R["PD_RF"] = rf_model.predict_proba(X_test_scaled)[:, 1]
df_R["PD_RF"] = rf_model.predict_proba(scaler.transform(X))[:, 1] # needs to be checked
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF
0,name0,55,1,1,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,...,False,False,False,False,False,False,False,0.679787,0.712740,0.24
1,name1,43,0,3,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,...,False,False,False,True,False,False,False,0.701779,0.707140,0.88
2,name2,46,0,0,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,...,False,False,False,True,False,False,False,0.125340,0.065778,0.25
3,name3,54,0,0,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,...,False,False,False,False,False,False,False,0.464511,0.419346,0.19
4,name4,36,1,0,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,...,False,True,False,False,False,False,False,0.869159,0.932222,0.93
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,1,1,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,...,True,False,True,False,False,False,False,0.642825,0.705934,0.21
996,name996,31,1,0,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,...,True,False,True,False,False,False,False,0.987255,0.946485,0.84
997,name997,59,3,1,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,...,False,False,False,True,False,False,False,0.892420,0.775484,0.87
998,name998,32,2,0,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,...,False,False,False,True,False,False,False,0.830169,0.857296,0.69


### Using Logistic LASSO

#### Packages

In [163]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

#### Parameter of the regularization strenght

In [164]:
alpha = 0.01  # Regularization strength

#### Training the regression

In [165]:
# L1-regularized logistic regression (like lasso but for classification problems)
logistic_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1/alpha, random_state=42)

# Fit the model
logistic_lasso.fit(X_train_scaled, y_train)


LogisticRegression(C=100.0, penalty='l1', random_state=42, solver='liblinear')

In [166]:
logistic_lasso.coef_

array([[ 0.01724507,  0.79745384,  0.011262  , -0.10350167, -0.06540044,
        -0.06981159,  0.52250618,  0.07578246,  0.43231641, -0.08368433,
         0.07409043,  0.16509792, -0.30240418,  0.33141434, -0.0503263 ,
         0.20652962, -0.00894937, -0.23373579, -0.21598324,  0.16505859,
        -0.02504131]])

#### Making predictions with the regression

In [167]:
# Predict probabilities
y_pred_proba = logistic_lasso.predict_proba(X_test_scaled)[:, 1]  # Probabilities of class 1

# Apply threshold at 0.5 to get predicted class labels
y_pred_class = (y_pred_proba >= 0.5).astype(int)

Getting accuracy measures

In [168]:
# Accuracy by rule of PD >= 0.5 default and PD < 0.5 not default
y_pred_class = (y_pred_proba >= 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {accuracy:.4f}")

# AUC score (for probability-based performance)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC: {auc:.4f}")

Accuracy: 0.6400
AUC: 0.7239


#### Now estiamting the PDs for our whole data set

In [169]:
df_R["PD_LL"] = logistic_lasso.predict_proba(scaler.transform(X))[:, 1]

### Now getting the general statistics of what I did, for all the four models: Logisitc regression, Neuronal networks, Random Forests and Logistic LASSO

In [170]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF,PD_LL
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.160000,0.490000,0.563000,3483.544000,3208.436846,275.107154,0.128476,3045.408715,0.863071,41.652000,0.845029,25696.412000,33469.095155,0.503000,0.503868,0.505903,0.504270,0.503871
std,8.892327,0.698848,0.823838,2572.014586,2442.556265,1666.772375,0.264127,2796.832486,0.361949,16.290865,0.499962,21419.236325,25418.158449,0.500241,0.223726,0.231212,0.324828,0.225432
min,30.000000,0.000000,0.000000,500.000000,397.434871,-7282.707532,0.000000,191.786562,0.250100,12.000000,0.000000,0.000000,91.300003,0.000000,0.087040,0.058590,0.010000,0.086211
25%,38.000000,0.000000,0.000000,1665.750000,1490.876937,-312.978644,0.000000,1259.758203,0.533992,24.000000,0.467627,8659.000000,13020.198067,0.000000,0.326561,0.319815,0.180000,0.324094
50%,45.000000,0.000000,0.000000,2583.000000,2410.963129,221.829460,0.000000,2121.094317,0.859688,48.000000,0.835855,21545.000000,27816.199018,1.000000,0.474448,0.489131,0.515000,0.472616
75%,53.000000,1.000000,1.000000,4500.000000,4145.065571,739.280135,0.158281,3695.929214,1.173887,60.000000,1.154614,40281.500000,49273.597159,1.000000,0.679925,0.700729,0.830000,0.679937
max,60.000000,3.000000,4.000000,14316.000000,17497.177007,11257.112312,2.220228,20405.717420,1.499850,60.000000,3.163049,79605.000000,119287.601837,1.000000,0.987255,0.967004,0.980000,0.987954


## EAD (exposure at default)
It is the total amount at risk at the moment the borrower defaults

That is the amount fo the credit that has been unpaid at the moment of the calculation

In [171]:
df_R["EAD"] = df_R["credit: monthly amount"] * 12
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF,PD_LL,EAD
0,name0,55,1,1,5999.0,5530.230019,468.769981,0.000000,2108.140876,0.840368,...,False,False,False,False,False,0.679787,0.712740,0.24,0.681669,25297.690515
1,name1,43,0,3,1809.0,3753.695941,-1944.695941,1.075012,1662.313632,0.521341,...,False,True,False,False,False,0.701779,0.707140,0.88,0.706186,19947.763582
2,name2,46,0,0,3278.0,2095.447533,1182.552467,0.000000,1585.662308,0.432012,...,False,True,False,False,False,0.125340,0.065778,0.25,0.126597,19027.947700
3,name3,54,0,0,9129.0,5727.409911,3401.590089,0.000000,5897.129297,1.355528,...,False,False,False,False,False,0.464511,0.419346,0.19,0.472492,70765.551562
4,name4,36,1,0,4120.0,6018.861015,-1898.861015,0.460889,2046.367546,0.654185,...,False,False,False,False,False,0.869159,0.932222,0.93,0.872254,24556.410551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,60,1,1,1533.0,2131.145946,-598.145946,0.390180,404.698011,0.386605,...,True,False,False,False,False,0.642825,0.705934,0.21,0.646762,4856.376131
996,name996,31,1,0,500.0,1384.567380,-884.567380,1.769135,696.957218,0.815367,...,True,False,False,False,False,0.987255,0.946485,0.84,0.987954,8363.486618
997,name997,59,3,1,4451.0,5292.689251,-841.689251,0.189101,2965.463907,0.621523,...,False,True,False,False,False,0.892420,0.775484,0.87,0.893636,35585.566888
998,name998,32,2,0,1769.0,1112.311114,656.688886,0.000000,2156.384596,1.428503,...,False,True,False,False,False,0.830169,0.857296,0.69,0.833199,25876.615153


## LGD (loss given default)

Example:

1. A borrower takes a loan of €10,000. 
2. They default after paying back €2,000, and the bank recovers €4,000 by seizing assets. 

That means:

Total Recovered = €2,000 (paid) + €4,000 (recovered from assets) = €6,000

Total Loss = €10,000 - €6,000 = €4,000

LGD = €4,000 (Total Loss)/ €10,000 (Loan Amount) = 40%



#### In our approximation 
LGD = (EAD - what can be sized)/EAD = 1 - (What can be seized/EAD)

In [172]:
def estimate_lgd(EAD, seizable_assets):
    lgd = 1 - (seizable_assets / EAD)
    return max(0, min(lgd, 1))  # The lgd should be between 0 and 1

In [173]:
df_R["LGD"] = df_R.apply(lambda row: estimate_lgd(row["EAD"], row["estimated seizable assets"]), axis=1) # This should be applied row by row

# We need here a Montecarlo to get the model with the highest accurracy results and they will be then added to the df, the others wont be added, but will remain part of the code.

## EL (expected loss)

EL=PD×LGD×EAD

In [174]:
df_R["EL"] = df_R["LGD"] * df_R["EAD"] * df_R["PD_NN"]

## We want here to estimate a suggested interest rate that should be charged to the customer

#### Inputs of the function

In [ ]:
base_rate = 0.03  # central bank or risk-free rate
max_rate = 0.45 # maximum legal rate to be charged
PD = df_R["PD_NN"]  # Probability of default
LGD = df_R["LGD"]  # Loss given default
bank_margin = 0.02  # Bank's margin

#### Function

In [ ]:
def calculate_suggested_rate(base_rate, max_rate, PD, LGD, margin):
    # Calculate the risk premium
    risk_premium = PD * LGD
    # Calculate the suggested rate
    suggested_rate = base_rate + risk_premium + margin
    return min(suggested_rate, max_rate)

#### Estimation of the calculated rate

In [ ]:
df_R["Suggested interest rate"] = df_R.apply(
    lambda row: calculate_suggested_rate(base_rate, max_rate, row["PD_NN"], row["LGD"], bank_margin), axis=1
)

## Making one last description of the columns

In [175]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,...,collateral,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF,PD_LL,EAD,LGD,EL
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.160000,0.490000,0.563000,3483.544000,3208.436846,275.107154,0.128476,3045.408715,0.863071,41.652000,...,25696.412000,33469.095155,0.503000,0.503868,0.505903,0.504270,0.503871,36544.904579,0.268946,5916.931837
std,8.892327,0.698848,0.823838,2572.014586,2442.556265,1666.772375,0.264127,2796.832486,0.361949,16.290865,...,21419.236325,25418.158449,0.500241,0.223726,0.231212,0.324828,0.225432,33561.989832,0.322879,10580.942574
min,30.000000,0.000000,0.000000,500.000000,397.434871,-7282.707532,0.000000,191.786562,0.250100,12.000000,...,0.000000,91.300003,0.000000,0.087040,0.058590,0.010000,0.086211,2301.438743,0.000000,0.000000
25%,38.000000,0.000000,0.000000,1665.750000,1490.876937,-312.978644,0.000000,1259.758203,0.533992,24.000000,...,8659.000000,13020.198067,0.000000,0.326561,0.319815,0.180000,0.324094,15117.098435,0.000000,0.000000
50%,45.000000,0.000000,0.000000,2583.000000,2410.963129,221.829460,0.000000,2121.094317,0.859688,48.000000,...,21545.000000,27816.199018,1.000000,0.474448,0.489131,0.515000,0.472616,25453.131798,0.075628,652.704977
75%,53.000000,1.000000,1.000000,4500.000000,4145.065571,739.280135,0.158281,3695.929214,1.173887,60.000000,...,40281.500000,49273.597159,1.000000,0.679925,0.700729,0.830000,0.679937,44351.150562,0.535414,7253.792527
max,60.000000,3.000000,4.000000,14316.000000,17497.177007,11257.112312,2.220228,20405.717420,1.499850,60.000000,...,79605.000000,119287.601837,1.000000,0.987255,0.967004,0.980000,0.987954,244868.609034,0.989083,73390.695819
